# Split the Version 4 Optuna database

This notebook splits `logistic_regression_Version_4.db` so that we can commit to github.
The original filename remains the active Optuna database for the other notebook. It keeps the newest trials and the best trial.

Older trials become independent SQLite databases named `_part2.db`, `_part3.db`, and so on.

In [1]:
import os
import sqlite3
from contextlib import closing
from pathlib import Path
from tempfile import TemporaryDirectory

DB = Path("logistic_regression_Version_4.db")
MAX_BYTES = 95 * 1024**2
CHILD_TABLES = ("trial_values", "trial_params", "trial_user_attributes",
                "trial_system_attributes", "trial_intermediate_values", "trial_heartbeats")

## Split and verify

SQLite first copies the database to temporary files. 

It then removes the unwanted trials and their related rows, compacts each file, and checks the results. 

In [2]:
if list(DB.parent.glob(DB.stem + "_part*.db")):
    raise FileExistsError("A numbered part already exists. Move it before running again.")

with TemporaryDirectory(dir=DB.parent) as temporary:
    with closing(sqlite3.connect(f"file:{DB.resolve()}?mode=ro", uri=True)) as source:
        if source.execute("PRAGMA integrity_check").fetchone()[0] != "ok":
            raise RuntimeError("Source database is damaged. No files were changed.")
        if source.execute("PRAGMA foreign_key_check").fetchone():
            raise RuntimeError("Source has broken foreign keys. No files were changed.")
        if source.execute("SELECT count(*) FROM studies").fetchone()[0] != 1:
            raise ValueError("This notebook expects one Optuna study.")

        trial_ids = [row[0] for row in source.execute("SELECT trial_id FROM trials ORDER BY trial_id")]
        direction = source.execute("SELECT direction FROM study_directions WHERE objective=0").fetchone()[0]
        order = "DESC" if direction == "MAXIMIZE" else "ASC"
        best_id = source.execute(
            "SELECT t.trial_id FROM trials t JOIN trial_values v USING(trial_id) "
            "WHERE t.state='COMPLETE' AND v.objective=0 AND v.value_type='FINITE' "
            f"ORDER BY v.value {order} LIMIT 1"
        ).fetchone()[0]

        max_parts = min(len(trial_ids), max(5, (DB.stat().st_size + MAX_BYTES - 1) // MAX_BYTES + 2))
        for count in range(2, max_parts + 1):
            groups = [set(trial_ids[i * len(trial_ids) // count:(i + 1) * len(trial_ids) // count])
                      for i in range(count)][::-1]
            groups[0].add(best_id)
            for group in groups[1:]:
                group.discard(best_id)

            temporary_parts = []
            for index, keep in enumerate(groups, start=1):
                path = Path(temporary) / f"attempt_{count}_part_{index}.db"
                with closing(sqlite3.connect(path)) as part:
                    source.backup(part)
                    part.execute("PRAGMA journal_mode=DELETE")
                    part.execute("PRAGMA foreign_keys=ON")
                    part.execute("CREATE TEMP TABLE remove_ids(trial_id INTEGER PRIMARY KEY)")
                    part.executemany("INSERT INTO remove_ids VALUES (?)",
                                     ((trial_id,) for trial_id in trial_ids if trial_id not in keep))
                    for table in CHILD_TABLES:
                        part.execute(f"DELETE FROM {table} WHERE trial_id IN (SELECT trial_id FROM remove_ids)")
                    part.execute("DELETE FROM trials WHERE trial_id IN (SELECT trial_id FROM remove_ids)")
                    part.executemany("UPDATE trials SET number=? WHERE trial_id=?",
                                     ((number, trial_id) for number, trial_id in enumerate(sorted(keep))))
                    part.commit()
                    part.execute("VACUUM")
                    if part.execute("PRAGMA integrity_check").fetchone()[0] != "ok":
                        raise RuntimeError(f"Integrity check failed for {path.name}")
                    if part.execute("PRAGMA foreign_key_check").fetchone():
                        raise RuntimeError(f"Foreign key check failed for {path.name}")
                    if part.execute("SELECT count(*) FROM trials").fetchone()[0] != len(keep):
                        raise RuntimeError(f"Trial count mismatch for {path.name}")
                temporary_parts.append(path)

            if all(path.stat().st_size <= MAX_BYTES for path in temporary_parts):
                break
        else:
            raise RuntimeError(f"Could not fit within {max_parts} parts. No files were changed.")

    archived = [DB.with_name(f"{DB.stem}_part{index}.db")
                for index in range(2, len(temporary_parts) + 1)]
    for temporary_path, final_path in zip(temporary_parts[1:], archived):
        os.replace(temporary_path, final_path)
    os.replace(temporary_parts[0], DB)

for path in [DB, *archived]:
    with closing(sqlite3.connect(f"file:{path.resolve()}?mode=ro", uri=True)) as part:
        count = part.execute("SELECT count(*) FROM trials").fetchone()[0]
    print(f"{path.name}: {count:,} trials, {path.stat().st_size / 1024**2:.1f} MiB")

logistic_regression_Version_4.db: 1,560 trials, 89.4 MiB
logistic_regression_Version_4_part2.db: 1,559 trials, 89.3 MiB
logistic_regression_Version_4_part3.db: 1,559 trials, 88.9 MiB
